In [3]:
from pyspark.sql import *
from pyspark.sql.functions import* 
from pyspark.sql.types import *
from pyspark.sql.window import Window

StatementMeta(, 5d1a1104-06e9-457c-96d1-29e5dc1a6951, 5, Finished, Available, Finished)

In [4]:
retailers_df = spark.table("Silver.dbo.retailers")

StatementMeta(, 5d1a1104-06e9-457c-96d1-29e5dc1a6951, 6, Finished, Available, Finished)

In [5]:
repayments_df = spark.table("Silver.dbo.repayments")

StatementMeta(, 5d1a1104-06e9-457c-96d1-29e5dc1a6951, 7, Finished, Available, Finished)

In [6]:
transactions_df = spark.table("Silver.dbo.transactions")

StatementMeta(, 5d1a1104-06e9-457c-96d1-29e5dc1a6951, 8, Finished, Available, Finished)

In [1]:
# ============================================================================
# CREDIT SCORING MODEL - FEATURE ENGINEERING FOR UNBANKED RETAILERS
# Goal: Near-zero default while maximizing market coverage
# ============================================================================

import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime, timedelta
from functools import reduce

print("=" * 80)
print("CREDIT SCORING - FEATURE ENGINEERING & RISK TIER ASSIGNMENT")
print("Goal: Identify creditworthy retailers with near-zero default risk")
print("=" * 80)

# ============================================================================
# CONFIGURATION
# ============================================================================

OBSERVATION_DATES = [
    "2024-07-15", "2024-07-25", "2024-08-01", "2024-08-10", "2024-08-20",
    "2024-09-01", "2024-09-15", "2024-09-30", "2024-10-15",
]

PERFORMANCE_WINDOW_DAYS = 45
DEFAULT_THRESHOLD_DAYS = 30
SERIOUS_DEFAULT_THRESHOLD = 60

print(f"\nConfiguration:")
print(f"  - Observation dates: {len(OBSERVATION_DATES)} snapshots")
print(f"  - Performance window: {PERFORMANCE_WINDOW_DAYS} days")
print(f"  - Default threshold: {DEFAULT_THRESHOLD_DAYS} days (moderate)")
print(f"  - Serious default: {SERIOUS_DEFAULT_THRESHOLD} days (severe)")

# ============================================================================
# LOAD DATA
# ============================================================================

print("\nLoading data...")
transactions = spark.table("Silver.dbo.silver_retailer_transactions")
retailers_master = spark.table("Silver.dbo.retailers")

print(f"✓ Loaded {transactions.count():,} transactions")
print(f"✓ Loaded {retailers_master.count():,} retailers")

# ============================================================================
# MACROECONOMIC DATA
# ============================================================================

macro_data = spark.createDataFrame([
    ("2024-07-01", 1550.0, 33.4, 850.0),
    ("2024-07-15", 1570.0, 33.5, 870.0),
    ("2024-07-25", 1585.0, 33.6, 885.0),
    ("2024-08-01", 1600.0, 32.2, 900.0),
    ("2024-08-10", 1615.0, 32.4, 910.0),
    ("2024-08-20", 1630.0, 32.6, 915.0),
    ("2024-09-01", 1650.0, 32.7, 920.0),
    ("2024-09-15", 1655.0, 32.8, 930.0),
    ("2024-09-30", 1665.0, 33.0, 940.0),
    ("2024-10-15", 1670.0, 33.1, 950.0),
], ["observation_date", "usd_ngn_rate", "inflation_rate_pct", "petrol_price_ngn"])

macro_data = macro_data.withColumn("observation_date", F.to_date("observation_date"))

# ============================================================================
# FEATURE ENGINEERING WITH CREDIT SCORING
# ============================================================================

def create_features_at_observation_date(transactions_df, retailers_df, observation_date_str):
    """
    Create features for credit scoring at a specific observation date.
    """
    
    observation_date = F.lit(observation_date_str).cast(DateType())
    
    # Calculate available history
    try:
        earliest_date = transactions_df.select(F.min("order_date")).first()[0]
    except:
        earliest_date = datetime.strptime(observation_date_str, "%Y-%m-%d").date()

    obs_date_parsed = datetime.strptime(observation_date_str, "%Y-%m-%d").date()
    days_of_history = (obs_date_parsed - earliest_date).days
    
    print(f"  - Days of history available: {days_of_history}")
    
    historical_txns = transactions_df.filter(F.col("order_date") < observation_date)
    
    window_recent = 30 if days_of_history > 30 else days_of_history
    window_medium = 60 if days_of_history > 60 else days_of_history
    
    recent_txns = historical_txns.filter(
        F.col("order_date") >= F.date_sub(observation_date, window_recent)
    )
    
    medium_txns = historical_txns.filter(
        F.col("order_date") >= F.date_sub(observation_date, window_medium)
    )
    
    all_history = historical_txns
    
    # ========================================================================
    # PAYMENT BEHAVIOR FEATURES
    # ========================================================================
    
    features_recent = recent_txns.groupBy("retailer_id").agg(
        F.count("transaction_id").alias("txn_count_recent"),
        F.sum("order_amount").alias("total_value_recent"),
        F.avg("order_amount").alias("avg_order_value_recent"),
        F.avg("days_late").alias("avg_days_late_recent"),
        F.max("days_late").alias("max_days_late_recent"),
        F.stddev("days_late").alias("stddev_days_late_recent"),
        
        # Payment performance rates
        F.avg(F.when(F.col("days_late") == 0, 1).otherwise(0)).alias("on_time_rate_recent"),
        F.avg(F.when(F.col("days_late") > 0, 1).otherwise(0)).alias("any_late_rate_recent"),
        F.avg(F.when(F.col("days_late") > 15, 1).otherwise(0)).alias("late_rate_recent"),
        F.avg(F.when(F.col("days_late") > 30, 1).otherwise(0)).alias("serious_late_rate_recent"),
        
        # Never had issues (important for creditworthiness)
        F.max(F.when(F.col("days_late") > 30, 1).otherwise(0)).alias("ever_seriously_late_recent"),
    )
    
    features_medium = medium_txns.groupBy("retailer_id").agg(
        F.count("transaction_id").alias("txn_count_medium"),
        F.avg("order_amount").alias("avg_order_value_medium"),
        F.avg("days_late").alias("avg_days_late_medium"),
        F.avg(F.when(F.col("days_late") > 15, 1).otherwise(0)).alias("late_rate_medium"),
        F.avg(F.when(F.col("days_late") == 0, 1).otherwise(0)).alias("on_time_rate_medium"),
    )
    
    features_lifetime = all_history.groupBy("retailer_id").agg(
        F.count("transaction_id").alias("txn_count_lifetime"),
        F.sum("order_amount").alias("total_value_lifetime"),
        F.avg("order_amount").alias("avg_order_value_lifetime"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
        F.avg("days_late").alias("avg_days_late_lifetime"),
        F.max("days_late").alias("max_days_late_lifetime"),
        F.stddev("order_amount").alias("stddev_order_value"),
        
        # Lifetime payment performance
        F.avg(F.when(F.col("days_late") == 0, 1).otherwise(0)).alias("on_time_rate_lifetime"),
        F.max(F.when(F.col("days_late") > 60, 1).otherwise(0)).alias("ever_defaulted_lifetime"),
        
        # Product diversity
        F.size(F.array_distinct(F.flatten(F.collect_list("product_categories_array")))).alias("unique_categories")
    )
    
    # ========================================================================
    # BEHAVIORAL TRENDS
    # ========================================================================
    
    trend_features = features_recent.join(features_medium, "retailer_id", "left").select(
        "retailer_id",
        
        # Deterioration (red flag)
        (F.col("avg_days_late_recent") / 
         F.when(F.col("avg_days_late_medium") > 0, F.col("avg_days_late_medium")).otherwise(1)
        ).alias("payment_deterioration_ratio"),
        
        (F.col("late_rate_recent") - F.col("late_rate_medium")).alias("late_rate_change"),
        
        # Activity trends
        (F.col("txn_count_recent") / 
         F.when(F.col("txn_count_medium") > 0, F.col("txn_count_medium")).otherwise(1) * 2
        ).alias("txn_velocity_ratio"),
        
        # Improvement (green flag)
        (F.col("on_time_rate_recent") - F.col("on_time_rate_medium")).alias("on_time_improvement"),
    )
    
    # ========================================================================
    # RECENCY & CONSISTENCY
    # ========================================================================
    
    recency_features = features_lifetime.select(
        "retailer_id",
        F.datediff(observation_date, F.col("last_order_date")).alias("days_since_last_order"),
        F.datediff(F.col("last_order_date"), F.col("first_order_date")).alias("customer_tenure_days"),
        (F.col("txn_count_lifetime") / 
         F.greatest(F.datediff(observation_date, F.col("first_order_date")) / 30.0, F.lit(1))
        ).alias("avg_orders_per_month"),
    )
    
    # ========================================================================
    # COMBINE ALL FEATURES
    # ========================================================================
    
    all_features = features_recent \
        .join(features_medium, "retailer_id", "full") \
        .join(features_lifetime, "retailer_id", "full") \
        .join(trend_features, "retailer_id", "left") \
        .join(recency_features, "retailer_id", "left")
    
    # Join static attributes
    retailer_static = retailers_df.select(
        "retailer_id",
        "owner_age",
        "owner_gender",
        "shop_type",
        "state",
        "urbanization_level",
        "years_in_business",
        "num_employees",
        "has_business_registration",
        "mobile_money_pattern",
        "credit_segment",
        "credit_limit",
    ).dropDuplicates(["retailer_id"])
    
    final_features = retailer_static.join(all_features, "retailer_id", "left")
    
    # ========================================================================
    # DERIVED FEATURES
    # ========================================================================
    
    final_features = final_features \
        .withColumn("credit_utilization", 
                    F.col("avg_order_value_recent") / 
                    F.when(F.col("credit_limit") > 0, F.col("credit_limit")).otherwise(1)) \
        .withColumn("formality_score",
                    (F.when(F.col("has_business_registration"), 0.5).otherwise(0) +
                     F.when(F.col("num_employees") >= 2, 0.3).otherwise(0) +
                     F.when(F.col("shop_type").isin(["Mini Mart", "Superette"]), 0.2).otherwise(0))) \
        .withColumn("mobile_money_score",
                    F.when(F.col("mobile_money_pattern") == "Heavy User", 1.0)
                     .when(F.col("mobile_money_pattern") == "Regular User", 0.7)
                     .when(F.col("mobile_money_pattern") == "Light User", 0.4)
                     .otherwise(0.1)) \
        .withColumn("is_new_customer", 
                    F.when((F.col("txn_count_lifetime").isNull()) | 
                          (F.col("txn_count_lifetime") == 0), 1).otherwise(0)) \
        .withColumn("gender_encoded", 
                    F.when(F.col("owner_gender") == "Female", 1).otherwise(0)) \
        .withColumn("urbanization_encoded",
                    F.when(F.col("urbanization_level") == "Urban", 2)
                     .when(F.col("urbanization_level") == "Peri-Urban", 1)
                     .otherwise(0)) \
        .withColumn("shop_type_encoded",
                    F.when(F.col("shop_type") == "Superette", 4)
                     .when(F.col("shop_type") == "Mini Mart", 3)
                     .when(F.col("shop_type") == "Provision Store", 2)
                     .when(F.col("shop_type") == "Market Stall", 1)
                     .otherwise(0)) \
        .withColumn("payment_consistency",
                    F.when(F.col("stddev_days_late_recent").isNull(), 1.0)
                     .otherwise(1.0 / (1.0 + F.col("stddev_days_late_recent"))))
    
    # Add observation date
    final_features = final_features \
        .withColumn("observation_date", observation_date) \
        .withColumn("days_of_history", F.lit(days_of_history))
    
    return final_features

# ============================================================================
# CREATE LABELS (MULTIPLE SEVERITY LEVELS)
# ============================================================================

def create_labels_for_observation_date(transactions_df, observation_date_str, 
                                       performance_window_days):
    """
    Create multiple label types for credit scoring.
    """
    
    observation_date = F.lit(observation_date_str).cast(DateType())
    performance_end = F.date_add(observation_date, performance_window_days)
    
    latest_date = transactions_df.select(F.max("order_date")).first()[0]
    obs_date_parsed = datetime.strptime(observation_date_str, "%Y-%m-%d").date()
    days_of_future = (latest_date - obs_date_parsed).days
    
    print(f"  - Days of future data available: {days_of_future}")
    
    future_txns = transactions_df.filter(
        (F.col("order_date") >= observation_date) &
        (F.col("order_date") < performance_end)
    )
    
    labels = future_txns.groupBy("retailer_id").agg(
        # Multiple severity levels
        F.max(F.when(F.col("days_late") > 60, 1).otherwise(0)).alias("serious_default"),
        F.max(F.when(F.col("days_late") > 30, 1).otherwise(0)).alias("moderate_default"),
        F.max(F.when(F.col("days_late") > 15, 1).otherwise(0)).alias("minor_late"),
        F.max(F.when(F.col("days_late") > 0, 1).otherwise(0)).alias("any_late"),
        
        # For regression target
        F.max("days_late").alias("max_days_late_future"),
        F.avg("days_late").alias("avg_days_late_future"),
        
        # Context
        F.count("transaction_id").alias("num_future_txns"),
    )
    
    labels = labels.withColumn("has_performance_data", F.lit(1))
    
    # Create composite label (for backward compatibility)
    labels = labels.withColumn("will_default", F.col("moderate_default"))
    
    return labels

# ============================================================================
# CREDIT SCORE CALCULATION (RULE-BASED)
# ============================================================================

def calculate_credit_score(features_df):
    """
    Calculate credit score (300-850 scale) based on behavioral features.
    This creates a baseline score that ML can refine.
    """
    
    scored = features_df.withColumn(
        "base_credit_score",
        
        # Start at 500 (neutral)
        F.lit(500) +
        
        # === POSITIVE FACTORS ===
        
        # Perfect/excellent payment history (max +200)
        F.when(F.col("on_time_rate_lifetime") >= 0.98, 200)
         .when(F.col("on_time_rate_lifetime") >= 0.95, 175)
         .when(F.col("on_time_rate_lifetime") >= 0.90, 150)
         .when(F.col("on_time_rate_lifetime") >= 0.85, 125)
         .when(F.col("on_time_rate_lifetime") >= 0.80, 100)
         .when(F.col("on_time_rate_lifetime") >= 0.70, 75)
         .otherwise(0) +
        
        # Transaction volume/loyalty (max +100)
        F.when(F.col("txn_count_lifetime") >= 100, 100)
         .when(F.col("txn_count_lifetime") >= 50, 80)
         .when(F.col("txn_count_lifetime") >= 30, 60)
         .when(F.col("txn_count_lifetime") >= 15, 40)
         .when(F.col("txn_count_lifetime") >= 5, 20)
         .otherwise(0) +
        
        # Low average lateness (max +80)
        F.when(F.col("avg_days_late_lifetime") <= 1, 80)
         .when(F.col("avg_days_late_lifetime") <= 3, 60)
         .when(F.col("avg_days_late_lifetime") <= 5, 40)
         .when(F.col("avg_days_late_lifetime") <= 10, 20)
         .otherwise(0) +
        
        # Payment consistency (max +40)
        (F.col("payment_consistency") * 40) +
        
        # Business formality (max +40)
        (F.col("formality_score") * 40) +
        
        # Tenure/relationship length (max +40)
        F.when(F.col("customer_tenure_days") >= 365, 40)
         .when(F.col("customer_tenure_days") >= 180, 30)
         .when(F.col("customer_tenure_days") >= 90, 20)
         .when(F.col("customer_tenure_days") >= 30, 10)
         .otherwise(0) +
        
        # === NEGATIVE FACTORS ===
        
        # Ever seriously late (max -200)
        F.when(F.col("ever_defaulted_lifetime") == 1, -200)
         .when(F.col("max_days_late_lifetime") > 45, -150)
         .when(F.col("max_days_late_lifetime") > 30, -100)
         .when(F.col("max_days_late_lifetime") > 20, -50)
         .otherwise(0) -
        
        # Recent deterioration (max -150)
        F.when(F.col("payment_deterioration_ratio") > 2.0, 150)
         .when(F.col("payment_deterioration_ratio") > 1.5, 100)
         .when(F.col("payment_deterioration_ratio") > 1.2, 50)
         .otherwise(0) -
        
        # High recent late rate (max -100)
        F.when(F.col("late_rate_recent") >= 0.5, 100)
         .when(F.col("late_rate_recent") >= 0.3, 75)
         .when(F.col("late_rate_recent") >= 0.2, 50)
         .when(F.col("late_rate_recent") >= 0.1, 25)
         .otherwise(0) -
        
        # Inactivity (max -60)
        F.when(F.col("days_since_last_order") > 90, 60)
         .when(F.col("days_since_last_order") > 60, 40)
         .when(F.col("days_since_last_order") > 30, 20)
         .otherwise(0) -
        
        # Recent serious late payment (max -100)
        F.when(F.col("ever_seriously_late_recent") == 1, 100)
         .otherwise(0)
    )
    
    # Cap score between 300-850
    scored = scored.withColumn(
        "credit_score",
        F.when(F.col("base_credit_score") > 850, 850)
         .when(F.col("base_credit_score") < 300, 300)
         .otherwise(F.col("base_credit_score"))
    )
    
    # Assign risk tiers
    scored = scored.withColumn(
        "risk_tier",
        F.when(F.col("credit_score") >= 750, "Platinum")
         .when(F.col("credit_score") >= 650, "Gold")
         .when(F.col("credit_score") >= 550, "Silver")
         .when(F.col("credit_score") >= 450, "Bronze")
         .otherwise("Copper")
    )
    
    # Recommended action
    scored = scored.withColumn(
        "credit_decision",
        F.when(F.col("risk_tier").isin(["Platinum", "Gold"]), "APPROVE")
         .when(F.col("risk_tier") == "Silver", "APPROVE_WITH_MONITORING")
         .when(F.col("risk_tier") == "Bronze", "CONDITIONAL_APPROVAL")
         .otherwise("DECLINE")
    )
    
    return scored

# ============================================================================
# GENERATE TRAINING DATA
# ============================================================================

print("\n" + "=" * 80)
print("GENERATING CREDIT SCORING DATASET")
print("=" * 80)

all_training_data = []

for obs_date in OBSERVATION_DATES:
    print(f"\nProcessing: {obs_date}")
    
    features = create_features_at_observation_date(
        transactions, 
        retailers_master, 
        obs_date
    )
    
    labels = create_labels_for_observation_date(
        transactions,
        obs_date,
        PERFORMANCE_WINDOW_DAYS
    )
    
    training_sample = features.join(labels, "retailer_id", "left")
    
    # Filter: only retailers with performance data and history
    training_sample = training_sample.filter(
        (F.col("has_performance_data") == 1) &
        (F.col("is_new_customer") == 0)
    )
    
    # Calculate credit scores
    training_sample = calculate_credit_score(training_sample)
    
    # Join macro data
    training_sample = training_sample.join(macro_data, "observation_date", "left")
    
    count = training_sample.count()
    print(f"  ✓ Created {count:,} samples")
    
    if count > 0:
        all_training_data.append(training_sample)

# Combine all snapshots
print("\nCombining all snapshots...")
final_training_data = reduce(lambda df1, df2: df1.union(df2), all_training_data)

print(f"\n✓ Total training samples: {final_training_data.count():,}")

# ============================================================================
# HANDLE NULLS
# ============================================================================

final_training_data = final_training_data.fillna({
    "txn_count_recent": 0,
    "txn_count_medium": 0,
    "txn_count_lifetime": 0,
    "avg_days_late_recent": 0,
    "avg_days_late_medium": 0,
    "avg_days_late_lifetime": 0,
    "late_rate_recent": 0,
    "late_rate_medium": 0,
    "on_time_rate_recent": 1.0,
    "on_time_rate_medium": 1.0,
    "on_time_rate_lifetime": 1.0,
    "total_value_recent": 0,
    "avg_order_value_recent": 0,
    "credit_utilization": 0,
    "payment_deterioration_ratio": 1.0,
    "late_rate_change": 0,
    "txn_velocity_ratio": 1.0,
    "on_time_improvement": 0,
    "payment_consistency": 1.0,
    "will_default": 0,
    "serious_default": 0,
    "moderate_default": 0,
    "minor_late": 0,
    "any_late": 0,
    "max_days_late_future": 0,
    "stddev_days_late_recent": 0,
    "unique_categories": 0,
    "ever_seriously_late_recent": 0,
    "ever_defaulted_lifetime": 0,
})

# ============================================================================
# SAVE TO GOLD TABLE
# ============================================================================

final_training_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_credit_scoring_features")

print(f"\n✓ Saved to gold_credit_scoring_features")

# ============================================================================
# COMPREHENSIVE STATISTICS
# ============================================================================

print("\n" + "=" * 80)
print("CREDIT SCORING DATASET STATISTICS")
print("=" * 80)

print("\n1. Risk Tier Distribution:")
final_training_data.groupBy("risk_tier").agg(
    F.count("*").alias("retailers"),
    F.avg("credit_score").alias("avg_score"),
    F.min("credit_score").alias("min_score"),
    F.max("credit_score").alias("max_score"),
).orderBy(F.desc("avg_score")).show(truncate=False)

print("\n2. Default Rates by Risk Tier:")
final_training_data.groupBy("risk_tier").agg(
    F.count("*").alias("total"),
    F.sum("moderate_default").alias("defaults_30d"),
    F.sum("serious_default").alias("defaults_60d"),
    (F.sum("moderate_default") / F.count("*") * 100).alias("default_rate_pct"),
    (F.sum("serious_default") / F.count("*") * 100).alias("serious_default_pct"),
).orderBy(F.desc("default_rate_pct")).show(truncate=False)

print("\n3. Credit Decision Distribution:")
final_training_data.groupBy("credit_decision").agg(
    F.count("*").alias("count"),
    (F.count("*") / F.lit(final_training_data.count()) * 100).alias("percentage"),
    F.avg("credit_score").alias("avg_score"),
    F.sum("moderate_default").alias("defaults"),
).orderBy("credit_decision").show(truncate=False)

print("\n4. Performance by Observation Date:")
final_training_data.groupBy("observation_date").agg(
    F.count("*").alias("samples"),
    F.avg("credit_score").alias("avg_score"),
    F.avg("moderate_default").alias("default_rate"),
    F.countDistinct(F.when(F.col("risk_tier") == "Platinum", F.col("retailer_id"))).alias("platinum"),
    F.countDistinct(F.when(F.col("risk_tier") == "Gold", F.col("retailer_id"))).alias("gold"),
).orderBy("observation_date").show(20, False)

print("\n5. Near-Zero Default Analysis (Platinum + Gold tiers):")
top_tier = final_training_data.filter(
    F.col("risk_tier").isin(["Platinum", "Gold"])
)

total = top_tier.count()
defaults = top_tier.filter(F.col("moderate_default") == 1).count()
serious = top_tier.filter(F.col("serious_default") == 1).count()

print(f"  Total Platinum + Gold retailers: {total:,}")
print(f"  Defaults (30+ days): {defaults} ({defaults/total*100:.2f}%)")
print(f"  Serious defaults (60+ days): {serious} ({serious/total*100:.2f}%)")
print(f"  Market coverage: {total/final_training_data.count()*100:.1f}%")

print("\n6. Feature Importance Indicators:")
final_training_data.select(
    F.corr("on_time_rate_lifetime", "credit_score").alias("on_time_corr"),
    F.corr("avg_days_late_lifetime", "credit_score").alias("lateness_corr"),
    F.corr("txn_count_lifetime", "credit_score").alias("volume_corr"),
    F.corr("payment_deterioration_ratio", "moderate_default").alias("deterioration_to_default"),
).show()

print("\n" + "=" * 80)
print("✓ COMPLETE - Ready for ML model training")
print("=" * 80)
print("\nNext steps:")
print("  1. Train GBT/RF model to predict credit_score")
print("  2. Validate Platinum/Gold tiers have <1% default rate")
print("  3. Deploy with monitoring for drift")
print("  4. Set credit limits by tier (Platinum: high, Gold: medium, etc.)")
print("=" * 80)

StatementMeta(, 180467dc-f196-4cad-967c-fa6ebc4995f8, 3, Finished, Available, Finished)

CREDIT SCORING - FEATURE ENGINEERING & RISK TIER ASSIGNMENT
Goal: Identify creditworthy retailers with near-zero default risk

Configuration:
  - Observation dates: 9 snapshots
  - Performance window: 45 days
  - Default threshold: 30 days (moderate)
  - Serious default: 60 days (severe)

Loading data...
✓ Loaded 75,347 transactions
✓ Loaded 10,000 retailers

GENERATING CREDIT SCORING DATASET

Processing: 2024-07-15
  - Days of history available: 0
  - Days of future data available: 168
  ✓ Created 0 samples

Processing: 2024-07-25
  - Days of history available: 10
  - Days of future data available: 158
  ✓ Created 102 samples

Processing: 2024-08-01
  - Days of history available: 17
  - Days of future data available: 151
  ✓ Created 249 samples

Processing: 2024-08-10
  - Days of history available: 26
  - Days of future data available: 142
  ✓ Created 449 samples

Processing: 2024-08-20
  - Days of history available: 36
  - Days of future data available: 132
  ✓ Created 700 samples

P

In [5]:
# ============================================================================
# STEP 4: CREDIT LIMIT ASSIGNMENT BY RISK TIER
# Assign appropriate credit limits based on predicted risk and behavior
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 80)
print("CREDIT LIMIT ASSIGNMENT BY RISK TIER")
print("=" * 80)

# ============================================================================
# LOAD PREDICTIONS
# ============================================================================

print("\nLoading credit score predictions...")
predictions = spark.table("gold_credit_score_predictions")
features = spark.table("gold_credit_scoring_features")

# Join to get full feature set
credit_data = predictions.join(
    features.select(
        "retailer_id",
        "observation_date", 
        "avg_order_value_lifetime",
        "total_value_lifetime",
        "txn_count_lifetime",
        "credit_limit",  # Current limit
        "avg_orders_per_month",
        "on_time_rate_lifetime",
        "shop_type",
        "urbanization_level",
        "years_in_business",
    ),
    ["retailer_id", "observation_date"],
    "left"
)

print(f"✓ Loaded {credit_data.count():,} records")

# ============================================================================
# BASE CREDIT LIMIT BY TIER
# ============================================================================

print("\n" + "=" * 80)
print("STEP 1: BASE CREDIT LIMITS BY TIER")
print("=" * 80)

# Define base limits (in Naira)
base_limits = {
    "Platinum": 500000,  # ₦500K base
    "Gold": 250000,      # ₦250K base
    "Silver": 100000,    # ₦100K base
    "Bronze": 50000,     # ₦50K base
    "Copper": 0,         # No credit
}

print("\nBase Credit Limits:")
for tier, limit in sorted(base_limits.items(), key=lambda x: -x[1]):
    print(f"  {tier}: ₦{limit:,}")

# Apply base limits
credit_data = credit_data.withColumn(
    "base_credit_limit",
    F.when(F.col("predicted_tier") == "Platinum", 500000)
     .when(F.col("predicted_tier") == "Gold", 250000)
     .when(F.col("predicted_tier") == "Silver", 100000)
     .when(F.col("predicted_tier") == "Bronze", 50000)
     .otherwise(0)
)

# ============================================================================
# ADJUSTMENT FACTORS
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: CALCULATE ADJUSTMENT FACTORS")
print("=" * 80)

credit_data = credit_data.withColumn(
    # Factor 1: Transaction history multiplier (1.0 - 2.0x)
    "history_multiplier",
    F.when(F.col("txn_count_lifetime") >= 100, 2.0)
     .when(F.col("txn_count_lifetime") >= 50, 1.8)
     .when(F.col("txn_count_lifetime") >= 30, 1.5)
     .when(F.col("txn_count_lifetime") >= 15, 1.3)
     .when(F.col("txn_count_lifetime") >= 5, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    # Factor 2: Payment perfection bonus (1.0 - 1.5x)
    "payment_bonus",
    F.when(F.col("on_time_rate_lifetime") >= 0.99, 1.5)
     .when(F.col("on_time_rate_lifetime") >= 0.95, 1.3)
     .when(F.col("on_time_rate_lifetime") >= 0.90, 1.2)
     .when(F.col("on_time_rate_lifetime") >= 0.85, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    # Factor 3: Business maturity (1.0 - 1.3x)
    "maturity_bonus",
    F.when(F.col("years_in_business") >= 10, 1.3)
     .when(F.col("years_in_business") >= 5, 1.2)
     .when(F.col("years_in_business") >= 3, 1.1)
     .otherwise(1.0)
)

credit_data = credit_data.withColumn(
    # Factor 4: Shop type multiplier (0.8 - 1.2x)
    "shop_type_multiplier",
    F.when(F.col("shop_type") == "Superette", 1.2)
     .when(F.col("shop_type") == "Mini Mart", 1.1)
     .when(F.col("shop_type") == "Provision Store", 1.0)
     .when(F.col("shop_type") == "Market Stall", 0.9)
     .otherwise(0.8)
)

credit_data = credit_data.withColumn(
    # Factor 5: Location (urban = higher)
    "location_multiplier",
    F.when(F.col("urbanization_level") == "Urban", 1.2)
     .when(F.col("urbanization_level") == "Peri-Urban", 1.0)
     .otherwise(0.9)
)

# Combine all multipliers
credit_data = credit_data.withColumn(
    "total_multiplier",
    F.col("history_multiplier") * 
    F.col("payment_bonus") * 
    F.col("maturity_bonus") * 
    F.col("shop_type_multiplier") *
    F.col("location_multiplier")
)

# ============================================================================
# CALCULATE FINAL CREDIT LIMIT
# ============================================================================

print("\n" + "=" * 80)
print("STEP 3: CALCULATE FINAL CREDIT LIMITS")
print("=" * 80)

credit_data = credit_data.withColumn(
    "calculated_limit",
    (F.col("base_credit_limit") * F.col("total_multiplier")).cast("long")
)

# Apply caps by tier (prevent excessive limits)
credit_data = credit_data.withColumn(
    "final_credit_limit",
    F.when(F.col("predicted_tier") == "Platinum", 
           F.least(F.col("calculated_limit"), F.lit(2000000)))  # Cap at ₦2M
     .when(F.col("predicted_tier") == "Gold", 
           F.least(F.col("calculated_limit"), F.lit(1000000)))  # Cap at ₦1M
     .when(F.col("predicted_tier") == "Silver", 
           F.least(F.col("calculated_limit"), F.lit(500000)))   # Cap at ₦500K
     .when(F.col("predicted_tier") == "Bronze", 
           F.least(F.col("calculated_limit"), F.lit(150000)))   # Cap at ₦150K
     .otherwise(0)
)

# Calculate utilization based on average order value
credit_data = credit_data.withColumn(
    "recommended_utilization",
    F.col("avg_order_value_lifetime") / F.col("final_credit_limit")
)

# ============================================================================
# APPROVAL DECISION WITH CREDIT LIMIT
# ============================================================================

credit_data = credit_data.withColumn(
    "credit_decision_detail",
    F.when(F.col("predicted_tier") == "Platinum", 
           F.concat(F.lit("APPROVE - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Gold", 
           F.concat(F.lit("APPROVE - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Silver", 
           F.concat(F.lit("APPROVE (Monitor) - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .when(F.col("predicted_tier") == "Bronze", 
           F.concat(F.lit("CONDITIONAL - ₦"), F.format_number(F.col("final_credit_limit"), 0)))
     .otherwise("DECLINE - No Credit")
)

# ============================================================================
# STATISTICS & VALIDATION
# ============================================================================

print("\n" + "=" * 80)
print("CREDIT LIMIT STATISTICS BY TIER")
print("=" * 80)

limit_stats = credit_data.groupBy("predicted_tier").agg(
    F.count("*").alias("retailers"),
    F.min("final_credit_limit").alias("min_limit"),
    F.avg("final_credit_limit").alias("avg_limit"),
    F.max("final_credit_limit").alias("max_limit"),
    F.sum("final_credit_limit").alias("total_credit_exposure"),
    F.avg("total_multiplier").alias("avg_multiplier"),
    F.avg("recommended_utilization").alias("avg_utilization"),
).orderBy(F.desc("avg_limit"))

print("\nCredit Limits by Tier:")
limit_stats.show(truncate=False)

# Total credit exposure
total_exposure = credit_data.agg(F.sum("final_credit_limit")).first()[0]
print(f"\nTotal Credit Exposure: ₦{total_exposure:,}")

# ============================================================================
# RISK-ADJUSTED CREDIT ALLOCATION
# ============================================================================

print("\n" + "=" * 80)
print("RISK-ADJUSTED CREDIT ALLOCATION")
print("=" * 80)

# Calculate expected loss based on historical default rates
credit_data = credit_data.withColumn(
    "expected_default_rate",
    F.when(F.col("predicted_tier") == "Platinum", 0.001)  # 0.1%
     .when(F.col("predicted_tier") == "Gold", 0.005)      # 0.5%
     .when(F.col("predicted_tier") == "Silver", 0.01)     # 1%
     .when(F.col("predicted_tier") == "Bronze", 0.03)     # 3%
     .otherwise(0.30)  # 30% for Copper
)

credit_data = credit_data.withColumn(
    "expected_loss",
    (F.col("final_credit_limit") * F.col("expected_default_rate")).cast("long")
)

portfolio_risk = credit_data.groupBy("predicted_tier").agg(
    F.sum("final_credit_limit").alias("total_exposure"),
    F.sum("expected_loss").alias("total_expected_loss"),
    (F.sum("expected_loss") / F.sum("final_credit_limit") * 100).alias("loss_rate_pct"),
    F.count("*").alias("retailers")
).orderBy(F.desc("total_exposure"))

print("\nPortfolio Risk Analysis:")
portfolio_risk.show(truncate=False)

total_expected_loss = credit_data.agg(F.sum("expected_loss")).first()[0]
print(f"\nTotal Expected Loss: ₦{total_expected_loss:,}")
print(f"Portfolio Loss Rate: {total_expected_loss/total_exposure*100:.3f}%")

# ============================================================================
# SAVE CREDIT LIMITS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING CREDIT LIMIT ASSIGNMENTS")
print("=" * 80)

# Select final output columns
credit_limits_output = credit_data.select(
    "retailer_id",
    "observation_date",
    "predicted_score",
    "predicted_tier",
    "base_credit_limit",
    "final_credit_limit",
    "credit_decision_detail",
    "total_multiplier",
    "history_multiplier",
    "payment_bonus",
    "maturity_bonus",
    "shop_type_multiplier",
    "location_multiplier",
    "expected_default_rate",
    "expected_loss",
    "recommended_utilization",
    "moderate_default",
)

# Save to gold table
credit_limits_output.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_credit_limits")

print("✓ Credit limits saved to gold_credit_limits")

# ============================================================================
# SAMPLE OUTPUT
# ============================================================================

print("\n" + "=" * 80)
print("SAMPLE CREDIT LIMIT ASSIGNMENTS")
print("=" * 80)

print("\nPlatinum Tier Examples:")
credit_data.filter(F.col("predicted_tier") == "Platinum") \
    .select(
        "retailer_id",
        "predicted_score",
        "final_credit_limit",
        "total_multiplier",
        "txn_count_lifetime",
        "on_time_rate_lifetime",
        "credit_decision_detail"
    ).orderBy(F.desc("final_credit_limit")).show(5, truncate=False)

print("\nGold Tier Examples:")
credit_data.filter(F.col("predicted_tier") == "Gold") \
    .select(
        "retailer_id",
        "predicted_score",
        "final_credit_limit",
        "total_multiplier",
        "txn_count_lifetime",
        "on_time_rate_lifetime",
        "credit_decision_detail"
    ).orderBy(F.desc("final_credit_limit")).show(5, truncate=False)

# ============================================================================
# BUSINESS RECOMMENDATIONS
# ============================================================================

print("\n" + "=" * 80)
print("BUSINESS RECOMMENDATIONS")
print("=" * 80)

# Scenario comparison
approved_only = credit_data.filter(
    F.col("predicted_tier").isin(["Platinum", "Gold"])
)

print("\nScenario: Approve Platinum + Gold Only")
print(f"  Total retailers: {approved_only.count():,}")
print(f"  Total credit exposure: ₦{approved_only.agg(F.sum('final_credit_limit')).first()[0]:,}")
print(f"  Expected loss: ₦{approved_only.agg(F.sum('expected_loss')).first()[0]:,}")
print(f"  Average limit per retailer: ₦{approved_only.agg(F.avg('final_credit_limit')).first()[0]:,.0f}")

# If including Silver
with_silver = credit_data.filter(
    F.col("predicted_tier").isin(["Platinum", "Gold", "Silver"])
)

print("\nScenario: Approve Platinum + Gold + Silver")
print(f"  Total retailers: {with_silver.count():,}")
print(f"  Total credit exposure: ₦{with_silver.agg(F.sum('final_credit_limit')).first()[0]:,}")
print(f"  Expected loss: ₦{with_silver.agg(F.sum('expected_loss')).first()[0]:,}")
print(f"  Average limit per retailer: ₦{with_silver.agg(F.avg('final_credit_limit')).first()[0]:,.0f}")

print("\n" + "=" * 80)
print("✓ CREDIT LIMIT ASSIGNMENT COMPLETE")
print("=" * 80)
print("\nNext Step: Build monitoring dashboard for drift detection")
print("=" * 80)

StatementMeta(, 180467dc-f196-4cad-967c-fa6ebc4995f8, 7, Finished, Available, Finished)

CREDIT LIMIT ASSIGNMENT BY RISK TIER

Loading credit score predictions...
✓ Loaded 4,668 records

STEP 1: BASE CREDIT LIMITS BY TIER

Base Credit Limits:
  Platinum: ₦500,000
  Gold: ₦250,000
  Silver: ₦100,000
  Bronze: ₦50,000
  Copper: ₦0

STEP 2: CALCULATE ADJUSTMENT FACTORS

STEP 3: CALCULATE FINAL CREDIT LIMITS

CREDIT LIMIT STATISTICS BY TIER

Credit Limits by Tier:
+--------------+---------+---------+------------------+---------+---------------------+------------------+-------------------+
|predicted_tier|retailers|min_limit|avg_limit         |max_limit|total_credit_exposure|avg_multiplier    |avg_utilization    |
+--------------+---------+---------+------------------+---------+---------------------+------------------+-------------------+
|Platinum      |52       |528000   |827947.4230769231 |1287000  |43053266             |1.655895          |0.02034572507522834|
|Gold          |387      |217800   |366244.70284237724|561600   |141736700            |1.4649788113695095|0.04726831